# 01 Data And Subsets
Rohdaten laden, mentions bauen, Stage-Subsets erzeugen, QC ausgeben.

In [1]:
from pathlib import Path
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
PATHS_CONFIG = "configs/paths.local.yaml"  # alternativ: configs/paths.colab.yaml

def _find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "src").exists() and (c / "configs").exists():
            return c.resolve()
    return start.resolve()

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

RUN_ID = None  # Optional override. Default: auto from notebook 00 latest_run.json


ROOT: /home/ubuntu/Author_Name_Disambiguation


In [2]:
import pandas as pd

from src.common.config import (
    load_notebook_context,
    build_run_dirs,
    resolve_run_id,
    write_run_consistency,
)
from src.data.prepare_lspo import prepare_lspo_mentions
from src.data.prepare_ads import prepare_ads_mentions
from src.common.subset_builder import build_stage_subset, write_subset_manifest
from src.common.io_schema import save_parquet, read_parquet
from src.common.run_report import summarize_block_distribution
from src.common.subset_artifacts import (
    atomic_save_parquet,
    compute_source_fp,
    compute_subset_identity,
    resolve_manifest_paths,
    resolve_shared_subset_paths,
)

CTX = load_notebook_context(run_stage=RUN_STAGE, project_root=ROOT, paths_config=PATHS_CONFIG)
DATA = CTX["DATA"]
ART = CTX["ART"]
RUN = CTX["RUN"]

latest_context_path = Path(ART["metrics_dir"]) / "latest_run.json"
RUN_ID = resolve_run_id(RUN_ID, latest_context_path, allow_placeholder=False)
RUN_DIRS = build_run_dirs(DATA, ART, RUN_ID)
for key in ["metrics", "subset_cache", "subset_manifests", "interim"]:
    RUN_DIRS[key].mkdir(parents=True, exist_ok=True)

write_run_consistency(
    run_id=RUN_ID,
    run_stage=RUN_STAGE,
    run_dirs=RUN_DIRS,
    output_path=RUN_DIRS["metrics"] / "01_run_consistency.json",
    extras={"notebook": "01_data_and_subsets", "latest_context_path": str(latest_context_path)},
)

print("RUN_ID:", RUN_ID)



RUN_ID: smoke_20260213T134837Z_f0fc32b8


In [3]:
raw_checks = {
    "raw_lspo_parquet": Path(DATA["raw_lspo_parquet"]).exists(),
    "raw_lspo_h5": Path(DATA["raw_lspo_h5"]).exists(),
    "raw_ads_publications": Path(DATA["raw_ads_publications"]).exists(),
    "raw_ads_references": Path(DATA["raw_ads_references"]).exists(),
}
raw_checks


{'raw_lspo_parquet': True,
 'raw_lspo_h5': False,
 'raw_ads_publications': False,
 'raw_ads_references': True}

In [4]:
FORCE_REBUILD_MENTIONS = False

lspo_mentions_path = RUN_DIRS["interim"] / "lspo_mentions.parquet"
ads_mentions_path = RUN_DIRS["interim"] / "ads_mentions.parquet"

if lspo_mentions_path.exists() and not FORCE_REBUILD_MENTIONS:
    lspo_mentions = read_parquet(lspo_mentions_path)
    print("LSPO mentions cache hit ->", lspo_mentions_path)
else:
    lspo_mentions = prepare_lspo_mentions(
        parquet_path=DATA["raw_lspo_parquet"],
        h5_path=DATA.get("raw_lspo_h5"),
        output_path=lspo_mentions_path,
    )
    print("LSPO mentions rebuilt ->", lspo_mentions_path)

if ads_mentions_path.exists() and not FORCE_REBUILD_MENTIONS:
    ads_mentions = read_parquet(ads_mentions_path)
    print("ADS mentions cache hit ->", ads_mentions_path)
else:
    ads_mentions = prepare_ads_mentions(
        publications_path=DATA["raw_ads_publications"],
        references_path=DATA["raw_ads_references"],
        output_path=ads_mentions_path,
    )
    print("ADS mentions rebuilt ->", ads_mentions_path)

print("LSPO mentions:", len(lspo_mentions))
print("ADS mentions:", len(ads_mentions))


LSPO mentions cache hit -> /home/ubuntu/Author_Name_Disambiguation/data/interim/lspo_mentions.parquet
ADS mentions cache hit -> /home/ubuntu/Author_Name_Disambiguation/data/interim/ads_mentions.parquet
LSPO mentions: 554962
ADS mentions: 1234223


In [5]:
import time

stage = RUN["stage"]
seed = int(RUN.get("seed", 11))
_raw_target = RUN.get("subset_target_mentions")
target = None if _raw_target is None else int(_raw_target)

FORCE_REBUILD_SUBSETS = False

source_fp = compute_source_fp(lspo_mentions_path, ads_mentions_path)
identity = compute_subset_identity(RUN, source_fp=source_fp, sampler_version="v2")
subset_paths = resolve_shared_subset_paths(DATA, identity)
manifest_paths = resolve_manifest_paths(
    run_id=RUN_ID,
    manifest_dir=RUN_DIRS["subset_manifests"],
    identity=identity,
    run_stage=RUN_STAGE,
)
subset_paths.shared_dir.mkdir(parents=True, exist_ok=True)

print("Subset tag:", identity.subset_tag)
print("Source fingerprint:", identity.source_fp)
print("Config fingerprint:", identity.cfg_fp)

cache_ready = subset_paths.lspo_shared.exists() and subset_paths.ads_shared.exists()
if cache_ready and not FORCE_REBUILD_SUBSETS:
    print("Subset cache decision: hit")
    t0 = time.perf_counter()
    lspo_subset = read_parquet(subset_paths.lspo_shared)
    t1 = time.perf_counter()
    ads_subset = read_parquet(subset_paths.ads_shared)
    t2 = time.perf_counter()
    print(f"Load subset cache (LSPO): {t1 - t0:.2f}s <- {subset_paths.lspo_shared}")
    print(f"Load subset cache (ADS): {t2 - t1:.2f}s <- {subset_paths.ads_shared}")
else:
    reason = "force_rebuild" if FORCE_REBUILD_SUBSETS else "cache_miss"
    print("Subset cache decision:", reason)

    t0 = time.perf_counter()
    lspo_subset = build_stage_subset(
        lspo_mentions,
        stage=stage,
        seed=seed,
        target_mentions=target,
        subset_sampling=RUN.get("subset_sampling", {}),
    )
    t1 = time.perf_counter()
    ads_subset = build_stage_subset(
        ads_mentions,
        stage=stage,
        seed=seed,
        target_mentions=target,
        subset_sampling=RUN.get("subset_sampling", {}),
    )
    t2 = time.perf_counter()

    atomic_save_parquet(lspo_subset, subset_paths.lspo_shared, index=False)
    t3 = time.perf_counter()
    atomic_save_parquet(ads_subset, subset_paths.ads_shared, index=False)
    t4 = time.perf_counter()

    print(f"Build subset (LSPO): {t1 - t0:.2f}s")
    print(f"Build subset (ADS): {t2 - t1:.2f}s")
    print(f"Save subset (LSPO): {t3 - t2:.2f}s -> {subset_paths.lspo_shared}")
    print(f"Save subset (ADS): {t4 - t3:.2f}s -> {subset_paths.ads_shared}")

if FORCE_REBUILD_SUBSETS or not manifest_paths.lspo_primary.exists():
    write_subset_manifest(lspo_subset, manifest_paths.lspo_primary)
if FORCE_REBUILD_SUBSETS or not manifest_paths.ads_primary.exists():
    write_subset_manifest(ads_subset, manifest_paths.ads_primary)

print("LSPO subset:", len(lspo_subset), "->", subset_paths.lspo_shared)
print("ADS subset:", len(ads_subset), "->", subset_paths.ads_shared)
print("LSPO manifest ->", manifest_paths.lspo_primary)
print("ADS manifest ->", manifest_paths.ads_primary)



Subset tag: smoke_seed11_target5000_src9ded9146c5be
Source fingerprint: lspo:426307667-1770979025692818563|ads:828122380-1770979350280165309
Subset cache decision: hit
Load subset cache (LSPO): 0.02s <- /home/ubuntu/Author_Name_Disambiguation/data/subsets/cache/_shared/lspo_mentions_smoke_seed11_target5000_src9ded9146c5be.parquet
Load subset cache (ADS): 0.03s <- /home/ubuntu/Author_Name_Disambiguation/data/subsets/cache/_shared/ads_mentions_smoke_seed11_target5000_src9ded9146c5be.parquet
LSPO subset: 5000 -> /home/ubuntu/Author_Name_Disambiguation/data/subsets/cache/_shared/lspo_mentions_smoke_seed11_target5000_src9ded9146c5be.parquet
ADS subset: 5000 -> /home/ubuntu/Author_Name_Disambiguation/data/subsets/cache/_shared/ads_mentions_smoke_seed11_target5000_src9ded9146c5be.parquet
LSPO manifest -> /home/ubuntu/Author_Name_Disambiguation/data/subsets/manifests/smoke_20260213T134837Z_f0fc32b8_lspo_smoke_seed11_target5000_src9ded9146c5be_manifest.parquet
ADS manifest -> /home/ubuntu/Autho

In [6]:
def qc_table(df, name):
    return {
        "name": name,
        "records": len(df),
        "mentions": df["mention_id"].nunique(),
        "bibcodes": df["bibcode"].nunique(),
        "blocks": df["block_key"].nunique(),
        "missing_title": int(df["title"].isna().sum() + (df["title"].astype(str).str.strip()=="").sum()),
        "missing_abstract": int(df["abstract"].isna().sum() + (df["abstract"].astype(str).str.strip()=="").sum()),
        "missing_year": int(df["year"].isna().sum()),
        "missing_author": int(df["author_raw"].isna().sum() + (df["author_raw"].astype(str).str.strip()=="").sum()),
    }

pd.DataFrame([
    qc_table(lspo_subset, "lspo_subset"),
    qc_table(ads_subset, "ads_subset"),
])


,name,records,mentions,bibcodes,blocks,missing_title,missing_abstract,missing_year,missing_author
0,lspo_subset,5000,5000,5000,2500,0,0,5000,0
1,ads_subset,5000,5000,4385,2500,2,612,0,0


In [7]:
print("Top-20 ambige LSPO-Blöcke")
display(summarize_block_distribution(lspo_subset, top_k=20))
print("Top-20 ambige ADS-Blöcke")
display(summarize_block_distribution(ads_subset, top_k=20))

def suspect_block_summary(df: pd.DataFrame, name: str) -> dict:
    blocks = df["block_key"].fillna("").astype(str)
    block_df = blocks.to_frame(name="block_key").drop_duplicates()

    surname_len = block_df["block_key"].str.split(".").str[-1].fillna("").str.len()
    short_surname_blocks = int((surname_len <= 2).sum())

    tmp = df[["block_key", "author_raw"]].copy()
    tmp["surname"] = tmp["author_raw"].fillna("").astype(str).str.lower().str.replace(r"[^a-z ]", "", regex=True).str.split().str[-1]
    surname_variants = tmp.groupby("block_key")["surname"].nunique(dropna=True)
    collision_blocks = int((surname_variants >= 4).sum())

    total_blocks = max(1, int(block_df["block_key"].nunique()))
    suspect_blocks = int(((surname_len <= 2).sum()) + collision_blocks)
    return {
        "dataset": name,
        "total_blocks": total_blocks,
        "short_surname_blocks": short_surname_blocks,
        "high_collision_blocks": collision_blocks,
        "suspect_block_ratio": suspect_blocks / total_blocks,
    }

display(pd.DataFrame([
    suspect_block_summary(lspo_subset, "lspo_subset"),
    suspect_block_summary(ads_subset, "ads_subset"),
]))


Top-20 ambige LSPO-Blöcke


,block_key,mention_count
0,a.adriani,2
1,a.aghakouchak,2
2,a.aghamohammadi,2
3,a.akimov,2
4,a.alastuey,2
5,a.algora,2
6,a.ali,2
7,a.amara,2
8,a.amato,2
9,a.amorim,2


Top-20 ambige ADS-Blöcke


,block_key,mention_count
0,a.anderson,2
1,a.ashtekar,2
2,a.baden,2
3,a.balachandran,2
4,a.ball,2
5,a.banerjee,2
6,a.barbarogaltieri,2
7,a.barnes,2
8,a.barut,2
9,a.beretvas,2


,dataset,total_blocks,short_surname_blocks,high_collision_blocks,suspect_block_ratio
0,lspo_subset,2500,69,0,0.0276
1,ads_subset,2500,120,0,0.0480
